# HeatGrid Copilot ML 프로젝트 보고서

이 보고서는 현재 프로젝트 폴더에 남아 있는 산출물만 사용해 `Isolation Forest + LightGBM + Priority Engine` 흐름을 보고용으로 정리한다.

보고서의 기준은 다음과 같다.

- 코드나 모델 학습 로직은 수정하지 않는다.
- `data/processed`, `model_handoff`, `PREPROCESSING/docs`에 존재하는 현재 산출물만 읽는다.
- 파기되었거나 공식 채택되지 않은 후보도 수치 비교에 포함한다.
- 모든 시각화는 Plotly로 구성하고, 제목과 축 라벨은 한국어로 표기한다.
- 표는 사람이 검토하기 쉽도록 주요 컬럼명을 한국어로 바꾸고 수치는 반올림한다.

In [1]:
from pathlib import Path
import json
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "report":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "PREPROCESSING" / "docs"
HANDOFF_DIR = PROJECT_ROOT / "model_handoff" / "heatgrid_ml_models_2026-06-25"
COLOR_SEQ = px.colors.qualitative.Set2

COLUMN_KO = {
    "model": "모델",
    "model_name": "모델명",
    "variant": "실험 variant",
    "experiment_name": "실험명",
    "bucket_mapping": "버킷 구조",
    "feature_family": "Feature family",
    "feature_count": "Feature 수",
    "label_count": "라벨 수",
    "split": "데이터 구간",
    "scope": "평가 범위",
    "metric_type": "평가 방식",
    "row_count": "전체 행 수",
    "normal_count": "정상 수",
    "pre_fault_count": "위험구간 수",
    "roc_auc": "ROC-AUC",
    "average_precision": "Average Precision",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "precision_high_or_critical": "Precision(High 이상)",
    "recall_high_or_critical": "Recall(High 이상)",
    "f1_high_or_critical": "F1(High 이상)",
    "false_positive_rate": "False Positive Rate",
    "false_positive_rate_high_or_critical": "FPR(High 이상)",
    "accuracy": "Accuracy",
    "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1",
    "top2_accuracy": "Top2 Accuracy",
    "bucket_distance_mae": "Bucket 거리 MAE",
    "threshold_quantile": "Threshold quantile",
    "threshold_value": "Threshold 값",
    "count": "건수",
    "priority_level": "우선순위 등급",
    "priority_score": "우선순위 점수",
    "engine_version": "엔진 버전",
    "total_gain_importance": "전체 gain 중요도",
    "total_split_importance": "전체 split 중요도",
    "mean_holdout_permutation": "Holdout permutation 평균 기여도",
    "positive_holdout_feature_count": "Holdout 양수 기여 feature 수",
    "feature": "Feature",
    "gain_importance": "Gain 중요도",
    "split_importance": "Split 중요도",
    "holdout_permutation": "Holdout permutation",
    "train_permutation": "Train permutation",
    "validation_permutation": "Validation permutation",
}

VALUE_KO = {
    "train": "학습",
    "validation": "검증",
    "holdout": "홀드아웃",
    "overall": "전체",
    "manufacturer_2_sh": "제조사2/SH 그룹",
    "base": "기본 기준",
    "calibrated": "보정 기준",
    "low": "낮음",
    "medium": "중간",
    "high": "높음",
    "urgent": "긴급",
}


def read_csv(rel_path: str) -> pd.DataFrame:
    path = PROJECT_ROOT / rel_path
    if not path.exists():
        print(f"파일 없음: {rel_path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json(rel_path: str) -> dict:
    path = PROJECT_ROOT / rel_path
    if not path.exists():
        print(f"파일 없음: {rel_path}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def ko_df(df: pd.DataFrame, columns=None, rename_extra=None, decimals=4) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if columns is not None:
        out = out[[c for c in columns if c in out.columns]].copy()
    rename_map = dict(COLUMN_KO)
    if rename_extra:
        rename_map.update(rename_extra)
    out = out.rename(columns={c: rename_map.get(c, c) for c in out.columns})
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].round(decimals)
        elif out[col].dtype == "object":
            out[col] = out[col].map(lambda x: VALUE_KO.get(x, x) if isinstance(x, str) else x)
    return out


def show_table(df: pd.DataFrame, title: str, columns=None, rename_extra=None, decimals=4, height=None):
    view = ko_df(df, columns=columns, rename_extra=rename_extra, decimals=decimals)
    if view.empty:
        print(f"표시할 데이터가 없습니다: {title}")
        return view
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(view.columns),
            fill_color="#2F5597",
            font=dict(color="white", size=12),
            align="left",
            height=30,
        ),
        cells=dict(
            values=[view[c] for c in view.columns],
            fill_color="#F7F9FC",
            align="left",
            height=28,
        ),
    )])
    fig.update_layout(title=title, height=height or min(780, 170 + 34 * len(view)))
    fig.show()
    return view

print("프로젝트 루트:", PROJECT_ROOT)

프로젝트 루트: C:\Project3\HeatGrid_Agent


## 1. 전체 구조 요약

현재 프로젝트의 공식 흐름은 다음 구조다.

1. 29개 operational base column을 유지한다.
2. windowing과 feature engineering으로 모델 입력 feature를 만든다.
3. Isolation Forest가 정상 패턴 대비 이상 정도를 `anomaly_score`로 산출한다.
4. LightGBM risk 모델이 고장신고 전 위험 패턴과의 유사도를 `risk_probability`로 산출한다.
5. LightGBM leadtime 모델이 신고 전 pseudo lead time bucket을 추정한다.
6. Priority Engine이 `anomaly + risk + leadtime + event history`를 합쳐 설비실별 점검 우선순위를 계산한다.

중요한 해석 원칙은 다음과 같다.

- Isolation Forest 결과는 고장 확정이 아니라 정상 패턴 대비 이탈 정도다.
- risk 모델 결과는 실제 고장 확률이라기보다 `faults.csv` 신고 전 위험 패턴과의 유사도다.
- leadtime 모델은 실제 고장 발생 시각 예측이 아니라 신고시점 기준 pseudo lead time bucket 추정이다.

In [2]:
stages = [
    "Raw operational\n50개",
    "전처리 유지\n29개",
    "Feature 후보\n259개",
    "04 선택 feature\n195개",
    "05 Isolation Forest\nanomaly_score",
    "06 LGBM Risk\n189개 입력",
    "06 LGBM Leadtime\n221개 입력",
    "07 Priority Engine\n우선순위 점수",
    "Agent 전달\n판단/설명",
]
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=18,
        thickness=18,
        label=stages,
        color=["#5B8FF9", "#61DDAA", "#65789B", "#F6BD16", "#7262FD", "#78D3F8", "#9661BC", "#F6903D", "#008685"],
    ),
    link=dict(
        source=[0, 1, 2, 3, 4, 5, 6, 7],
        target=[1, 2, 3, 4, 5, 6, 7, 8],
        value=[50, 29, 259, 195, 195, 189, 221, 29],
        color="rgba(120,120,120,0.25)",
    )
)])
fig.update_layout(title="HeatGrid Copilot ML 전체 처리 흐름", font=dict(size=13), height=520)
fig.show()

## 2. 데이터 계약과 feature 수

보고용으로 혼동하기 쉬운 숫자는 29, 195, 189, 221이다.

- 29개: raw operational에서 불필요 컬럼을 제거하고 전처리 후 유지하는 base column 수다.
- 195개: 04 feature selection 이후 baseline 모델 입력으로 고정된 feature 수다.
- 189개: 공식 risk 모델 입력 feature 수다. 일부 누적 meter 계열 등은 risk 모델에서 제외됐다.
- 221개: leadtime promoted 모델 입력 feature 수다. risk/anomaly 결과와 timeflow 파생 feature가 추가됐기 때문에 195보다 많다.

따라서 221개는 raw column 수가 아니라 leadtime 모델 전용 입력 feature 수다.

In [3]:
feature_meta = read_csv("data/processed/ml_features/feature_columns.csv")
feature_family = read_csv("data/processed/ml_features/feature_family_summary.csv")
anomaly_meta = read_json("model_handoff/heatgrid_ml_models_2026-06-25/anomaly/baseline_model_metadata.json")
risk_meta = read_json("model_handoff/heatgrid_ml_models_2026-06-25/risk/risk_model_metadata.json")
lead_meta = read_json("model_handoff/heatgrid_ml_models_2026-06-25/leadtime/leadtime_bucket_model_promoted_metadata.json")

selected_count = int(feature_meta["selected_for_baseline"].astype(str).str.lower().eq("true").sum()) if not feature_meta.empty else 195
counts = pd.DataFrame([
    {"구분": "Raw operational 전체", "개수": 50, "설명": "원본 operational sensor/control column"},
    {"구분": "전처리 후 유지 base", "개수": 29, "설명": "불필요 raw column 제거 후 유지"},
    {"구분": "Feature 후보 전체", "개수": len(feature_meta), "설명": "feature_columns.csv 전체"},
    {"구분": "04 선택 feature", "개수": selected_count, "설명": "selected_for_baseline=True"},
    {"구분": "Isolation Forest 입력", "개수": anomaly_meta.get("feature_count", 195), "설명": "정상 패턴 기반 anomaly score"},
    {"구분": "Risk 모델 입력", "개수": risk_meta.get("feature_count", 189), "설명": "LightGBM binary risk"},
    {"구분": "Leadtime 모델 입력", "개수": lead_meta.get("feature_count", 221), "설명": "LightGBM 3-bucket + timeflow"},
])
fig = px.bar(
    counts,
    x="구분",
    y="개수",
    text="개수",
    color="구분",
    title="단계별 column / feature 수 비교",
    labels={"구분": "처리 단계", "개수": "개수"},
    color_discrete_sequence=COLOR_SEQ,
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_range=[0, max(counts["개수"]) * 1.2], height=500)
fig.show()
show_table(counts, "단계별 column / feature 수 요약", height=430)

,구분,개수,설명
0,Raw operational 전체,50,원본 operational sensor/control column
1,전처리 후 유지 base,29,불필요 raw column 제거 후 유지
2,Feature 후보 전체,259,feature_columns.csv 전체
3,04 선택 feature,195,selected_for_baseline=True
4,Isolation Forest 입력,195,정상 패턴 기반 anomaly score
5,Risk 모델 입력,189,LightGBM binary risk
6,Leadtime 모델 입력,221,LightGBM 3-bucket + timeflow


### feature family 분포

04 이후 모델 입력은 단일 센서값만 쓰는 구조가 아니다. 시간 주기, one-hot 상태값, event context, sensor 통계가 함께 들어간다. 이 구조 덕분에 LightGBM은 단순 수치 이상뿐 아니라 계절성, 운전 모드, 이벤트 이력도 같이 볼 수 있다.

In [4]:
if not feature_family.empty:
    fig = px.pie(
        feature_family,
        names="feature_family",
        values="feature_count",
        title="04 선택 feature family 구성",
        hole=0.35,
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_traces(textposition="inside", textinfo="percent+label")
    fig.show()
    show_table(feature_family, "04 선택 feature family 표", height=420)
else:
    print("feature_family_summary.csv를 찾지 못했습니다.")

## 3. 05 Isolation Forest 이상탐지

Isolation Forest는 정상 운전 패턴과 다른 구간을 찾는 비지도 모델이다. 이 단계는 고장을 확정하지 않고, 이후 risk/leadtime 모델이 사용할 `anomaly_score`를 만든다.

현재 threshold는 train normal score 분포의 상위 quantile 기준으로 잡혀 있으며, threshold를 낮추면 recall은 올라갈 수 있지만 false positive도 커질 수 있다.

In [5]:
if_metrics = read_csv("data/processed/ml_baseline/anomaly_baseline_metrics.csv")
if_sweep = read_csv("data/processed/ml_baseline/anomaly_baseline_threshold_sweep_metrics.csv")
metric_cols = ["roc_auc", "average_precision", "precision", "recall", "f1", "false_positive_rate"]
label_map = {
    "roc_auc": "ROC-AUC",
    "average_precision": "Average Precision",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "false_positive_rate": "False Positive Rate",
}

hold_if = if_metrics[(if_metrics["evaluation_split_column"] == "split_time_based") & (if_metrics["split"] == "holdout")].copy()
if not hold_if.empty:
    hold_long = hold_if.melt(id_vars=["split", "threshold_quantile"], value_vars=metric_cols, var_name="지표", value_name="값")
    hold_long["지표"] = hold_long["지표"].map(label_map)
    fig = px.bar(
        hold_long,
        x="지표",
        y="값",
        color="지표",
        title="Isolation Forest holdout 성능 요약",
        labels={"값": "점수", "지표": "평가지표"},
        text=hold_long["값"].map(lambda x: f"{x:.3f}"),
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(showlegend=False, yaxis_range=[0, 1.05], height=460)
    fig.show()
    show_table(
        hold_if,
        "Isolation Forest holdout 지표 표",
        columns=["model_name", "threshold_quantile", "threshold_value", "row_count", "normal_count", "pre_fault_count", *metric_cols],
        height=360,
    )

if not if_sweep.empty:
    sweep = if_sweep[if_sweep["evaluation_split_column"] == "split_time_based"].copy()
    sweep_long = sweep.melt(
        id_vars=["split", "threshold_quantile"],
        value_vars=["recall", "f1", "false_positive_rate"],
        var_name="지표",
        value_name="값",
    )
    sweep_long["지표"] = sweep_long["지표"].map({"recall": "Recall", "f1": "F1", "false_positive_rate": "False Positive Rate"})
    fig = px.line(
        sweep_long,
        x="threshold_quantile",
        y="값",
        color="지표",
        line_dash="split",
        markers=True,
        title="Isolation Forest threshold quantile 변화에 따른 성능",
        labels={"threshold_quantile": "Threshold quantile", "값": "점수", "지표": "지표", "split": "Split"},
    )
    fig.update_layout(height=520)
    fig.show()
    show_table(
        sweep[sweep["split"].eq("holdout")],
        "Isolation Forest holdout threshold sweep 표",
        columns=["threshold_quantile", "threshold_value", "precision", "recall", "f1", "false_positive_rate", "roc_auc", "average_precision"],
        height=420,
    )

## 4. 06 Risk 모델 비교

Risk 모델은 LightGBM binary classifier다. 목표는 실제 고장 확정이 아니라 `faults.csv` 신고 전 위험구간과 유사한 패턴인지 판단하는 것이다.

현재 공식본은 `lgbm_risk_scores_calibrated.csv`이며, group calibration으로 false positive를 줄이는 방향을 채택했다. promoted risk 후보는 실험상 의미는 있었지만 공식본을 교체할 수준은 아니어서 보류됐다.

In [6]:
risk_base = read_csv("data/processed/ml_risk/lgbm_risk_metrics.csv")
risk_cal = read_csv("data/processed/ml_risk/lgbm_risk_metrics_calibrated.csv")
risk_prom = read_csv("data/processed/ml_risk/lgbm_risk_metrics_promoted.csv")
rows = []
if not risk_base.empty:
    b = risk_base[(risk_base["group_name"] == "split_event_regime_based") & (risk_base["group_value"] == "holdout")]
    if not b.empty:
        r = b.iloc[0].to_dict(); r["모델"] = "risk 기본"; rows.append(r)
if not risk_cal.empty:
    c = risk_cal[(risk_cal["group_name"] == "split_event_regime_based") & (risk_cal["group_value"] == "holdout")]
    if not c.empty:
        r = c.iloc[0].to_dict(); r["모델"] = "risk calibrated 공식"; rows.append(r)
if not risk_prom.empty:
    p = risk_prom[(risk_prom["split"] == "holdout") & (risk_prom["scope"] == "overall")]
    for _, rr in p.iterrows():
        d = rr.to_dict(); d["모델"] = f"promoted 후보 {rr['metric_type']}"; rows.append(d)

risk_compare = pd.DataFrame(rows)
metrics = ["roc_auc", "average_precision", "precision_high_or_critical", "recall_high_or_critical", "f1_high_or_critical", "false_positive_rate_high_or_critical"]
if not risk_compare.empty:
    long = risk_compare.melt(id_vars=["모델"], value_vars=[m for m in metrics if m in risk_compare.columns], var_name="지표", value_name="값")
    long["지표"] = long["지표"].map({
        "roc_auc": "ROC-AUC",
        "average_precision": "Average Precision",
        "precision_high_or_critical": "Precision",
        "recall_high_or_critical": "Recall",
        "f1_high_or_critical": "F1",
        "false_positive_rate_high_or_critical": "FPR",
    })
    fig = px.bar(
        long,
        x="지표",
        y="값",
        color="모델",
        barmode="group",
        title="Risk 모델 holdout 성능 비교",
        labels={"값": "점수", "지표": "평가지표", "모델": "모델 버전"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(yaxis_range=[0, 1.05], height=560)
    fig.show()
    show_table(
        risk_compare,
        "Risk 모델 holdout 성능 비교 표",
        columns=["모델", "row_count", "normal_count", "pre_fault_count", *metrics],
        height=430,
    )

### Risk 개선 실험: event context와 thermal feature

성능 개선 과정에서는 event context, thermal relation, combined feature 후보를 비교했다. 현재 결론은 다음과 같다.

- event context는 holdout에서 안정적인 기여가 확인됐다.
- thermal relation은 일부 문제 그룹에서는 false positive를 줄이는 효과가 있었지만 전체 공식본을 교체할 수준은 아니었다.
- combined 후보도 의미는 있었지만 risk 공식본은 calibrated 유지가 더 안정적이라고 판단했다.

In [7]:
event_cmp = read_csv("data/processed/ml_risk/lgbm_event_context_comparison.csv")
state_holdout = read_csv("data/processed/ml_risk/lgbm_event_context_state_experiment_holdout.csv")
thermal_holdout = read_csv("data/processed/ml_risk/lgbm_thermal_feature_experiment_holdout.csv")
combined_holdout = read_csv("data/processed/ml_risk/lgbm_combined_feature_experiment_holdout.csv")

if not event_cmp.empty:
    h = event_cmp[event_cmp["group_value"] == "holdout"].copy()
    delta_cols = ["roc_auc_delta", "average_precision_delta", "precision_high_or_critical_delta", "recall_high_or_critical_delta", "f1_high_or_critical_delta", "false_positive_rate_high_or_critical_delta"]
    h_long = h.melt(value_vars=delta_cols, var_name="지표", value_name="개선폭")
    h_long["지표"] = h_long["지표"].map({
        "roc_auc_delta": "ROC-AUC",
        "average_precision_delta": "Average Precision",
        "precision_high_or_critical_delta": "Precision",
        "recall_high_or_critical_delta": "Recall",
        "f1_high_or_critical_delta": "F1",
        "false_positive_rate_high_or_critical_delta": "FPR",
    })
    fig = px.bar(
        h_long,
        x="지표",
        y="개선폭",
        color="개선폭",
        title="Event context 추가 전후 holdout 성능 변화",
        labels={"개선폭": "event_days_v1 - guarded_v1", "지표": "평가지표"},
        color_continuous_scale="RdBu",
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(height=460)
    fig.show()
    show_table(h_long, "Event context 추가 전후 holdout 성능 변화 표", height=430)

experiment_frames = []
for name, df in [("event 상태형", state_holdout), ("thermal 관계형", thermal_holdout), ("combined", combined_holdout)]:
    if not df.empty:
        tmp = df[(df["split"] == "holdout") & (df["scope"] == "overall")].copy()
        if tmp.empty:
            tmp = df[df["split"] == "holdout"].copy()
        tmp["실험군"] = name
        experiment_frames.append(tmp)

if experiment_frames:
    exp = pd.concat(experiment_frames, ignore_index=True)
    cols = ["실험군", "variant", "feature_count", "roc_auc", "precision_high_or_critical", "recall_high_or_critical", "f1_high_or_critical", "false_positive_rate_high_or_critical"]
    exp_view = exp[[c for c in cols if c in exp.columns]].sort_values("f1_high_or_critical", ascending=False).head(15)
    fig = px.scatter(
        exp_view,
        x="false_positive_rate_high_or_critical",
        y="f1_high_or_critical",
        color="실험군",
        size="feature_count",
        hover_name="variant",
        title="Risk 개선 후보: F1과 FPR 동시 비교",
        labels={"false_positive_rate_high_or_critical": "False Positive Rate", "f1_high_or_critical": "F1", "feature_count": "Feature 수", "실험군": "실험 종류"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(height=560)
    fig.show()
    show_table(exp_view, "Risk 개선 후보 상위 15개 표", height=700)

### Risk feature importance audit

feature importance audit에서는 train gain만 보지 않고 holdout permutation 기여도를 같이 봤다. 이 기준에서는 event context가 가장 안정적인 family로 나타났고, sensor numeric은 개수는 많지만 평균 holdout 기여도는 낮았다.

즉 다음 개선은 센서 절대값을 무작정 늘리는 것보다 event context 재표현, drift 후보 제거, 관계형 thermal feature 검증이 더 타당하다.

In [8]:
fi_family = read_csv("data/processed/ml_risk/lgbm_feature_importance_family_summary.csv")
fi_drift = read_csv("data/processed/ml_risk/lgbm_feature_importance_drift_candidates.csv")
if not fi_family.empty:
    fig = px.bar(
        fi_family.sort_values("mean_holdout_permutation", ascending=True),
        x="mean_holdout_permutation",
        y="feature_family",
        orientation="h",
        color="mean_holdout_permutation",
        title="Risk feature family별 holdout permutation 평균 기여도",
        labels={"mean_holdout_permutation": "Holdout permutation 평균 기여도", "feature_family": "Feature family"},
        color_continuous_scale="Teal",
    )
    fig.update_layout(height=440)
    fig.show()
    show_table(fi_family, "Risk feature family별 중요도 표", height=450)
if not fi_drift.empty:
    show_cols = [c for c in ["feature", "feature_family", "gain_importance", "split_importance", "holdout_permutation", "train_permutation", "validation_permutation"] if c in fi_drift.columns]
    show_table(fi_drift[show_cols].head(10), "Drift 의심 feature 상위 10개 표", height=520)

## 5. 06 Leadtime 모델 비교

Leadtime 모델은 pre_fault row 안에서 신고 시점까지의 pseudo lead time bucket을 추정한다. 현재 공식본은 3버킷 promoted 모델이다.

- `0-24h`
- `1-3d`
- `3-7d`

4버킷 구조는 holdout에서 악화되어 파기했고, 2버킷 구조는 macro F1은 좋지만 메인 leadtime 대체가 아니라 urgency 보조체인 후보로 보는 것이 맞다.

In [9]:
lead_base = read_csv("data/processed/ml_leadtime/leadtime_bucket_metrics.csv")
lead_prom = read_csv("data/processed/ml_leadtime/leadtime_bucket_metrics_promoted.csv")
lead_timeflow = read_csv("data/processed/ml_leadtime/leadtime_timeflow_experiment_holdout.csv")
lead_bucket = read_csv("data/processed/ml_leadtime/leadtime_bucket_redesign_experiment_holdout.csv")

lead_rows = []
for name, df in [("leadtime 기본", lead_base), ("leadtime promoted 공식", lead_prom)]:
    if not df.empty:
        h = df[df["split"] == "holdout"].copy()
        if not h.empty:
            r = h.iloc[0].to_dict(); r["모델"] = name; lead_rows.append(r)
lead_compare = pd.DataFrame(lead_rows)
lead_metrics = ["accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"]
if not lead_compare.empty:
    long = lead_compare.melt(id_vars=["모델"], value_vars=lead_metrics, var_name="지표", value_name="값")
    long["지표"] = long["지표"].map({"accuracy": "Accuracy", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1", "top2_accuracy": "Top2 Accuracy", "bucket_distance_mae": "Bucket MAE"})
    fig = px.bar(
        long,
        x="지표",
        y="값",
        color="모델",
        barmode="group",
        title="Leadtime 모델 holdout 성능 비교",
        labels={"값": "점수", "지표": "평가지표", "모델": "모델"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(height=520)
    fig.show()
    show_table(lead_compare, "Leadtime 모델 holdout 성능 비교 표", columns=["모델", "row_count", *lead_metrics], height=360)

if not lead_timeflow.empty:
    fig = px.line(
        lead_timeflow.sort_values("feature_count"),
        x="feature_count",
        y="macro_f1",
        text="experiment_name",
        markers=True,
        title="Leadtime timeflow feature 보강에 따른 holdout Macro F1",
        labels={"feature_count": "Feature 수", "macro_f1": "Macro F1", "experiment_name": "실험명"},
    )
    fig.update_traces(textposition="top center")
    fig.update_layout(height=500)
    fig.show()
    show_table(
        lead_timeflow[["experiment_name", "feature_count", "accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"]],
        "Leadtime timeflow 실험 holdout 표",
        height=420,
    )

In [10]:
if not lead_bucket.empty:
    bucket_long = lead_bucket.melt(
        id_vars=["experiment_name", "feature_count", "label_count", "bucket_mapping"],
        value_vars=["accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"],
        var_name="지표",
        value_name="값",
    )
    bucket_long["지표"] = bucket_long["지표"].map({"accuracy": "Accuracy", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1", "top2_accuracy": "Top2 Accuracy", "bucket_distance_mae": "Bucket MAE"})
    fig = px.bar(
        bucket_long,
        x="bucket_mapping",
        y="값",
        color="지표",
        barmode="group",
        title="Leadtime bucket 구조별 holdout 성능 비교",
        labels={"bucket_mapping": "Bucket 구조", "값": "점수", "지표": "평가지표"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(height=560)
    fig.show()
    show_table(
        lead_bucket[["experiment_name", "feature_count", "label_count", "accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"]],
        "Leadtime bucket 구조별 holdout 표",
        height=420,
    )

## 6. 07 Priority Engine

Priority Engine은 모델이 직접 고장을 진단하는 단계가 아니다. `risk + leadtime + anomaly + 최근 이벤트 이력`을 합쳐 지금 점검 우선순위를 산출하는 rule-based scoring layer다.

v1은 urgent 쏠림이 컸고, tuned v2는 urgent 포화를 줄이면서 high/medium 구간을 조금 더 분리했다. 현재 추천 출력은 tuned v2다.

In [11]:
prio_v1 = read_csv("data/processed/ml_priority/priority_engine_scores.csv")
prio_v2 = read_csv("data/processed/ml_priority/priority_engine_scores_tuned.csv")
prio_frames = []
for version, df in [("v1 baseline", prio_v1), ("v2 tuned 공식", prio_v2)]:
    if not df.empty:
        cnt = df["priority_level"].value_counts().rename_axis("priority_level").reset_index(name="count")
        cnt["버전"] = version
        prio_frames.append(cnt)
if prio_frames:
    prio_counts = pd.concat(prio_frames, ignore_index=True)
    order = ["urgent", "high", "medium", "low"]
    prio_counts["priority_level"] = pd.Categorical(prio_counts["priority_level"], categories=order, ordered=True)
    fig = px.bar(
        prio_counts.sort_values("priority_level"),
        x="priority_level",
        y="count",
        color="버전",
        barmode="group",
        text="count",
        title="Priority Engine 버전별 우선순위 등급 분포",
        labels={"priority_level": "우선순위 등급", "count": "건수", "버전": "엔진 버전"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(height=500)
    fig.show()
    show_table(prio_counts.sort_values(["버전", "priority_level"]), "Priority Engine 등급 분포 표", height=470)

score_frames = []
for version, df in [("v1 baseline", prio_v1), ("v2 tuned 공식", prio_v2)]:
    if not df.empty:
        tmp = df[["priority_score", "priority_level"]].copy()
        tmp["버전"] = version
        score_frames.append(tmp)
if score_frames:
    scores = pd.concat(score_frames, ignore_index=True)
    fig = px.box(
        scores,
        x="버전",
        y="priority_score",
        color="버전",
        points=False,
        title="Priority score 분포 비교",
        labels={"priority_score": "Priority score", "버전": "엔진 버전"},
        color_discrete_sequence=COLOR_SEQ,
    )
    fig.update_layout(height=460, showlegend=False)
    fig.show()
    score_summary = scores.groupby("버전")["priority_score"].describe().reset_index()
    show_table(score_summary, "Priority score 요약 통계 표", height=360)

## 7. 공식 채택 / 보류 / 파기 판단 요약

아래 표는 현재 산출물 기준으로 어떤 모델 또는 실험을 공식 흐름에 남겼고, 어떤 것은 보류 또는 파기했는지 정리한 것이다.

In [12]:
decisions = pd.DataFrame([
    {"항목": "Isolation Forest anomaly baseline", "판단": "채택", "근거": "정상 패턴 대비 이탈 정도를 anomaly_score로 제공. risk/leadtime 입력으로 사용.", "대표 수치": "holdout ROC-AUC 0.7152"},
    {"항목": "Risk LightGBM calibrated", "판단": "공식 채택", "근거": "group calibration으로 FPR을 낮추면서 holdout F1 유지.", "대표 수치": "F1 0.5466, FPR 0.1449, ROC-AUC 0.7628"},
    {"항목": "Risk promoted 후보", "판단": "보류", "근거": "실험 의미는 있으나 공식 calibrated 본을 교체할 수준은 아님.", "대표 수치": "공식본 대비 안정성 부족"},
    {"항목": "Leadtime 3버킷 promoted", "판단": "공식 채택", "근거": "timeflow_lag_delta_roll3가 Macro F1을 소폭 개선.", "대표 수치": "Accuracy 0.6512, Macro F1 0.4405, Top2 0.9651"},
    {"항목": "Leadtime 4버킷", "판단": "파기", "근거": "bucket을 세분화하자 holdout 정확도와 Macro F1이 악화.", "대표 수치": "Accuracy 0.5814, Macro F1 0.3432"},
    {"항목": "Leadtime 2버킷 urgency", "판단": "보조 후보", "근거": "Macro F1은 좋지만 메인 3버킷을 대체하면 정보량이 줄어듦.", "대표 수치": "Macro F1 0.6120"},
    {"항목": "Priority Engine v2 tuned", "판단": "공식 채택", "근거": "v1의 urgent 쏠림을 줄이고 high/medium 구간 분해력 개선.", "대표 수치": "urgent 514 -> 316"},
    {"항목": "Paper-aligned Autoencoder 계열", "판단": "legacy 보관", "근거": "현재 공식 흐름은 IF + LightGBM이며 autoencoder 계열은 비교/참고 자료로 분리.", "대표 수치": "paper_aligned 산출물 별도 보관"},
])
show_table(decisions, "공식 채택 / 보류 / 파기 판단 요약", height=620)

,항목,판단,근거,대표 수치
0,Isolation Forest anomaly baseline,채택,정상 패턴 대비 이탈 정도를 anomaly_score로 제공. risk/leadti...,holdout ROC-AUC 0.7152
1,Risk LightGBM calibrated,공식 채택,group calibration으로 FPR을 낮추면서 holdout F1 유지.,"F1 0.5466, FPR 0.1449, ROC-AUC 0.7628"
2,Risk promoted 후보,보류,실험 의미는 있으나 공식 calibrated 본을 교체할 수준은 아님.,공식본 대비 안정성 부족
3,Leadtime 3버킷 promoted,공식 채택,timeflow_lag_delta_roll3가 Macro F1을 소폭 개선.,"Accuracy 0.6512, Macro F1 0.4405, Top2 0.9651"
4,Leadtime 4버킷,파기,bucket을 세분화하자 holdout 정확도와 Macro F1이 악화.,"Accuracy 0.5814, Macro F1 0.3432"
5,Leadtime 2버킷 urgency,보조 후보,Macro F1은 좋지만 메인 3버킷을 대체하면 정보량이 줄어듦.,Macro F1 0.6120
6,Priority Engine v2 tuned,공식 채택,v1의 urgent 쏠림을 줄이고 high/medium 구간 분해력 개선.,urgent 514 -> 316
7,Paper-aligned Autoencoder 계열,legacy 보관,현재 공식 흐름은 IF + LightGBM이며 autoencoder 계열은 비교/참...,paper_aligned 산출물 별도 보관


## 8. 최종 해석

현재 프로젝트는 `Isolation Forest + LightGBM risk + LightGBM leadtime + Priority Engine` 구조로 정리되어 있다. 이 구조는 다음 목적에는 부합한다.

- 정상 패턴과 다른 이상징후 탐지
- 이상징후가 고장신고 전 위험 패턴과 유사한지 판단
- 신고까지의 pseudo lead time bucket 추정
- 설비실별 점검 우선순위 산출

다만 현재 한계도 명확하다.

- ML은 실제 고장 발생 시각을 직접 예측하지 않는다.
- risk는 고장 확정 확률이 아니라 신고 전 위험 패턴 유사도다.
- leadtime은 실제 고장 발생 시각이 아니라 신고시점 기준 pseudo bucket이다.
- holdout 성능은 운영 가능한 1차 기준선 수준이며, 추가 개선은 feature 표현과 label 구조 개선이 필요하다.

다음 개선의 우선순위는 다음과 같다.

1. risk false negative audit 강화
2. drift 의심 feature 제거/축소 ablation
3. event context 상태형 재표현
4. thermal relation/group feature 보강
5. leadtime timeflow 확장
6. urgency 2버킷 보조체인 검토

즉, 다음 단계는 모델 종류를 바꾸는 것이 아니라 현재 계약을 유지한 상태에서 `feature 표현`, `오탐/미탐 audit`, `pseudo label 해석`을 정교화하는 것이다.